# Standard-star calibration quality vs. linphi positioners

Companion to `linphi_splitflux.ipynb`, using a different data source suggested by the
data-systems team: `calibstars-<expid>.csv`, the per-exposure standard-star flux
calibration table. `linphi_splitflux.ipynb` compares flux *repeatability* across exposures
for ordinary point sources; this notebook asks a related but different question -- among
the stars actually used for **flux calibration**, is there a systematic difference in
calibration quality between linphi-affected positioners and regular ones?

Three things this notebook has to establish before that comparison means anything:

1. **Confirm the calibstars fibers are genuinely standard-star targets.** `calibstars.csv`
   only *documents* itself as standard stars -- worth checking against an independent
   source rather than assuming. `OBJTYPE` in the FIBERMAP turns out not to be the right
   signal (it's `'TGT'` for every science target, standard star or not). The real signal is
   the `STD_FAINT`/`STD_WD`/`STD_BRIGHT` bits in `DESI_TARGET` for main DARK/BRIGHT-survey
   exposures -- **but** `BACKUP`-program exposures (bright/nearby targets for poor
   conditions) flag their standard stars in `MWS_TARGET` instead
   (`GAIA_STD_FAINT`/`GAIA_STD_WD`/`GAIA_STD_BRIGHT`), a real gotcha caught by running this
   over a wide selection and finding an exposure that failed the check 0/297 -- not
   hypothetical. Both masks happen to use the identical bit positions, just in different
   columns, so the same hardcoded bit constant (`STD_BITS`, from
   `desitarget.targetmask.desi_mask`/`mws_mask` -- not imported, just 3 known integers, to
   avoid the heavier `desitarget` dependency) is checked against both.
2. **Connect `calibstars`'s `FIBER` index to `(PETAL_LOC, DEVICE_LOC)`.** `calibstars` is
   indexed by `FIBER` (the whole-focal-plane 0-4999 numbering), not `(PETAL_LOC,
   DEVICE_LOC)` like `coords`/`cframe_table`. `FIBER // 500 == PETAL_LOC` always holds, but
   `DEVICE_LOC` is a scrambled hardware positioner ID with no formula -- confirmed by
   inspecting real data. `Exposure.fiberassign_table` (one file, whole focal plane) has
   `FIBER`, `(PETAL_LOC, DEVICE_LOC)`, `DESI_TARGET`, *and* `MWS_TARGET` together -- much
   faster than looping `cframe_table` over every petal for the same columns (confirmed
   ~60x: ~0.14s vs. ~9s per exposure).
3. **Attach the linphi flag**, exactly as in `linphi_splitflux.ipynb`: `Exposure.coords`,
   joined on the same `(PETAL_LOC, DEVICE_LOC)` index.

See `API.md`'s "Standard-star flux calibration table" and "FIBERASSIGN table" sections, and
"Known gotchas", for the FIBER-vs-(PETAL_LOC, DEVICE_LOC) distinction in more detail.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Add telemetry_mining to the path. Point DOS_TELEMETRY_MINING_DIR at your own
# checkout (svn co https://desi.lbl.gov/svn/code/online/telemetry_mining/trunk);
# defaults to the maintainer's checkout at NERSC.
TM_DIR = os.getenv("DOS_TELEMETRY_MINING_DIR", os.path.expanduser("~/telemetry_mining-trunk/src"))
sys.path.insert(0, TM_DIR)
from telemetry_mining import Exposure, select_exposures, harvest

## Exposure selection

Unlike `linphi_splitflux.ipynb`, this notebook doesn't difference anything between
exposures of the *same* tile -- each calibstars measurement stands on its own, pooled
across exposures at the end. That means a broad, multi-tile, multi-night selection is
entirely appropriate here (no same-tile requirement to worry about), so this is where the
general exposure-selection logic belongs, not the split-specific notebook.

Set **exactly one** of `NIGHT_RANGE` or `EXPIDS` below, plus `SEQUENCE` (a list, for
flexibility -- e.g. `['DESI', '_Split']` also picks up split follow-up exposures, which are
tagged `sequence='_Split'` rather than `'DESI'`; for this study plain `['DESI']` is what we
actually want, since we're not comparing splits to each other) and `MIN_TEFF` (minimum
accumulated effective time, in seconds, from the ETC's real-time `totteff` column --
excludes aborted/junk exposures that never accumulated meaningful signal).

In [ ]:
# Set exactly one of these -- leave the other as None
NIGHT_RANGE = [20260101, 20260710]            # e.g. (20240920, 20240930), inclusive
EXPIDS = None                                 # e.g. an explicit exposure list

SEQUENCE = ['DESI']   # regular DESI science exposures (not split follow-ups, for this study)
MIN_TEFF = 30.0       # minimum accumulated effective time (s), from the ETC's real-time totteff

assert (NIGHT_RANGE is None) != (EXPIDS is None), "set exactly one of NIGHT_RANGE or EXPIDS, not both/neither"

if NIGHT_RANGE is not None:
    where = "sequence = ANY(%s) and night between %s and %s and totteff > %s"
    params = (SEQUENCE, NIGHT_RANGE[0], NIGHT_RANGE[1], MIN_TEFF)
else:
    where = "sequence = ANY(%s) and id = ANY(%s) and totteff > %s"
    params = (SEQUENCE, list(EXPIDS), MIN_TEFF)

expids = list(select_exposures(where, params=params)['EXPID'])

# STD_FAINT (bit 33) | STD_WD (bit 34) | STD_BRIGHT (bit 35) from desitarget.targetmask.desi_mask
# (0x200000000 | 0x400000000 | 0x800000000) -- hardcoded rather than importing desitarget (a much
# heavier DESI-stack dependency), for 3 known integer bit constants. Both DESI_TARGET and the
# BACKUP-program MWS_TARGET use these same bit positions.
STD_BITS = 0xE00000000
print(f'{len(expids)} exposures selected')


## Per exposure: calibstars + STD confirmation + petal/device location + linphi flag

For each exposure:
- `exp.calibstars` -- the standard-star flux calibration table, indexed by FIBER.
- `exp.fiberassign_table` -- **one file, whole focal plane** (5000 rows), carrying FIBER
  plus DESI_TARGET alongside the (PETAL_LOC, DEVICE_LOC) index. This replaces an earlier,
  much slower version of this cell that looped `cframe_table('r{petal}')` over every
  petal (~0.9s per petal-read x up to 10 petals per exposure) just to get the same
  columns -- confirmed ~60x faster (~0.14s vs. ~9s) since `fiberassign_table` needs only
  one small file, not ten cframe files with large spectral arrays we don't use at all.
- Confirm every calibstars fiber has a STD bit set in `DESI_TARGET` -- printed per exposure,
  not silently assumed. Anything that *doesn't* match is printed explicitly rather than
  dropped quietly, since that would be a real anomaly worth looking at, not routine.
- Join in `PETAL_LOC`/`DEVICE_LOC` (from the fiberassign read above) and `POS_LINPHI` (from
  `exp.coords`, same join key as `linphi_splitflux.ipynb`).

In [ ]:
def calibstars_with_linphi(e):
    """Per-exposure: the calibstars table joined to targeting (to confirm STD) and the POS_LINPHI
    robot flag. Returns None to skip an exposure -- harvest drops those -- since calibstars-<expid>.csv
    or fiberassign-<tileid>.fits.gz can be missing even when the exposure passes the selection."""
    cs = e.calibstars
    fa = e.fiberassign_table
    if cs is None or fa is None:
        return None

    fiber_meta = fa.reset_index().set_index('FIBER')[['PETAL_LOC', 'DEVICE_LOC', 'DESI_TARGET', 'MWS_TARGET']]
    joined = cs.join(fiber_meta)

    # Main-survey standard stars are flagged in DESI_TARGET; BACKUP-program ones in MWS_TARGET
    # (GAIA_STD_*), which uses the identical bit positions -- so STD_BITS applies to either column.
    joined['is_std'] = (((joined['DESI_TARGET'].astype('int64') & STD_BITS) != 0) |
                        ((joined['MWS_TARGET'].astype('int64') & STD_BITS) != 0))
    if not joined['is_std'].all():   # sanity: every calibstars fiber should be a standard star
        bad = int((~joined['is_std']).sum())
        print(f'  exposure {e.expid}: {bad}/{len(joined)} calibstars fibers lack a STD bit -- inspect')

    # attach the linphi flag (coords is indexed by PETAL_LOC/DEVICE_LOC)
    return joined.join(e.coords[['POS_LINPHI']], on=['PETAL_LOC', 'DEVICE_LOC'])


# harvest pools ONE DataFrame across all exposures: it bulk-resolves `night`, inserts an `EXPID`
# column, drops exposures where the fn returned None, and reads them in parallel (max_workers).
# Threads are safe here -- each fn does light per-exposure I/O (calibstars CSV, fiberassign FITS,
# coords); nothing spawns its own process pool (unlike cframe_tables -- see fiberflux_ratio_linphi).
calib_all = harvest(expids, calibstars_with_linphi, concat=True, max_workers=8)
print(f'{len(calib_all)} calibstars measurements across {calib_all["EXPID"].nunique()} exposures')


## Regular vs. linphi robots: RCALIBFRAC comparison

Same string-vs-bool gotcha as `linphi_splitflux.ipynb`: `POS_LINPHI` is the literal string
`'True'`/`'False'`, not a real boolean.

**Sample-size caveat, stated plainly**: only ~6% of positioners are linphi-flagged overall,
and each exposure has roughly 100-300 standard stars -- so the linphi group is always a
small fraction of the total. Check the printed counts below before reading too much into
the comparison; widen `NIGHT_RANGE`/loosen `MIN_TEFF` above if the linphi sample looks too
thin.

### A note on `RCALIBFRAC` (and reprocessing)

`RCALIBFRAC` is normalized per exposure, and its **construction changed around mid-2025** -- a
point-source flat->psf aperture correction was added to the pipeline (see `API.md`'s data-uniformity
caveat and `MEASURED_VS_EXPECTED_FLUX.md`). This study is a **relative** regular-vs-linphi comparison
*within the same exposures*, so that change affects both robot groups equally and the conclusion
(linphi = more scatter, no bias) is robust. But the **absolute** `RCALIBFRAC` distribution shifts
across the mid-2025 boundary, so don't compare it across that date, and prefer a single uniformly-
reprocessed data release (not `daily`) for any long-baseline trend.

In [ ]:
regular = calib_all[calib_all['POS_LINPHI'] == 'False']
linphi = calib_all[calib_all['POS_LINPHI'] == 'True']
print(f'{len(regular)} regular-robot calibstar measurements, {len(linphi)} linphi-robot calibstar measurements')

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.hist(regular['RCALIBFRAC'], bins=30, range=(0.5, 1.5), color='b', alpha=0.6)
plt.title(f'RCALIBFRAC (regular robots), n={len(regular)}')
plt.subplot(1, 2, 2)
plt.hist(linphi['RCALIBFRAC'], bins=30, range=(0.5, 1.5), color='r', alpha=0.6)
plt.title(f'RCALIBFRAC (linphi robots), n={len(linphi)}')
plt.show()

print(f"mean RCALIBFRAC: regular={regular['RCALIBFRAC'].mean():.4f}, linphi={linphi['RCALIBFRAC'].mean():.4f}")
print(f"std  RCALIBFRAC: regular={regular['RCALIBFRAC'].std():.4f}, linphi={linphi['RCALIBFRAC'].std():.4f}")

## Recap

What this notebook adds on top of `linphi_splitflux.ipynb`:

| Question | How |
|---|---|
| Are calibstars fibers really standard stars? | `DESI_TARGET` bits for main-survey exposures, `MWS_TARGET` bits for `BACKUP`-program exposures (both hardcoded from `desitarget`'s masks to avoid the dependency) -- confirmed at scale, including a real BACKUP exposure that only passes via `MWS_TARGET` |
| How do I get PETAL_LOC/DEVICE_LOC (and targeting bits) for a FIBER-indexed table, fast? | `Exposure.fiberassign_table` (one whole-focal-plane file) -- ~60x faster than looping `cframe_table` over every petal for the same columns |
| Does calibration quality differ for linphi robots? | `RCALIBFRAC` split by `POS_LINPHI`, same join as `linphi_splitflux.ipynb` |

Exposure selection (`NIGHT_RANGE`/`EXPIDS` + `SEQUENCE`/`MIN_TEFF`, via `select_exposures`)
lives in this notebook rather than `linphi_splitflux.ipynb`, since nothing here differences
between exposures of the same tile -- see the "Exposure selection" cell above for why.

### Result (2026-07-17, `NIGHT_RANGE = [20260101, 20260710]`, full main-survey year to date)

3091 exposures selected, 3034 successfully processed (57 skipped -- no `calibstars`/
`fiberassign` file for that exposure, a real gap at this scale, not a bug: some overlap
with known DESI downtime periods, some likely individual missing-product cases). All
processed exposures came back 100% STD-confirmed.

|                | regular robots | linphi robots |
|---|---|---|
| n measurements | 420,874        | 17,386         |
| mean RCALIBFRAC | 0.9995        | 1.0008         |
| std RCALIBFRAC | 0.0556         | 0.0703         |

**Mean is essentially identical (no systematic offset, confirmed even more precisely than
the smaller sample below); linphi robots show ~27% more scatter.** Same signature as the
smaller sample, now at ~0.8% precision (`1/sqrt(17386)`) on the linphi side -- past the
~1% precision target from the smaller-sample result below. This is the expected signature
of an imprecise-but-unbiased positioning fault: the linphi issue degrades how precisely a
positioner centers a star on its fiber (less flux down the fiber on average, in either
direction), rather than pulling flux systematically high or low.